In [1]:
import pandas as pd

In [2]:
test_df = pd.read_csv("C:\\Users\\VP678WV\\OneDrive - EY\\Documents\\Delivery_Delay\\data\\processed\\test_encoded3.csv")

In [3]:
X_test = test_df.drop('target', axis=1)

In [14]:
X_test.columns

Index(['profit_per_order', 'order_item_discount', 'order_item_product_price',
       'order_item_profit_ratio', 'order_item_quantity', 'sales',
       'order_profit_per_order', 'shipping_mode', 'distance_normalized',
       'order_to_shipment_days', 'order_shipping_time',
       'order_to_shipment_planned_days', 'shipment_delay_days',
       'performance_score_order_full_location',
       'performance_score_customer_full_location',
       'performance_score_order_dayofweek',
       'performance_score_shipping_dayofweek', 'performance_score_order_hour',
       'performance_score_shipping_hour', 'performance_score_order_daynight',
       'performance_score_ship_daynight', 'payment_type_CASH',
       'payment_type_DEBIT', 'payment_type_PAYMENT', 'payment_type_TRANSFER'],
      dtype='object')

In [4]:
import joblib

In [6]:
model_pruned_c = joblib.load("rf_model_pruned.joblib")

In [24]:
import numpy as np
import shap
import random

CLASS_NAMES = {
    0: "early",
    1: "on_time",
    2: "delay"
}

def explain_random_sample(model, X, class_names=CLASS_NAMES, num_features=2):
    """
    Pick a random sample, predict with model, and explain using SHAP.
    """
    # ---- 1. Pick a random sample ----
    sample_idx = random.randint(0, len(X)-1)
    x_sample = X.iloc[[sample_idx]]

    # ---- 2. Predict ----
    proba = model.predict_proba(x_sample)[0]
    pred_class_idx = np.argmax(proba)

    # Map prediction to readable label
    pred_label = class_names.get(pred_class_idx, str(pred_class_idx))

    # ---- 3. SHAP explanation ----
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(x_sample)

    # print(shap_values)

    # Case A: list (older SHAP version with multiclass)
    if isinstance(shap_values, list):
        shap_for_pred = shap_values[pred_class_idx][0]
    # Case B: array (new SHAP version with shape (1, n_features, n_classes))
    elif shap_values.ndim == 3:
        shap_for_pred = shap_values[0, :, pred_class_idx]
    # Case C: binary (2D array (1, n_features))
    else:
        shap_for_pred = shap_values[0]

    # ---- 4. Top +ve and -ve features ----
    feature_importance = dict(zip(X.columns, shap_for_pred.tolist()))
    sorted_features = sorted(feature_importance.items(), key=lambda x: x[1], reverse=True)

    top_positive = [{f: v} for f, v in sorted_features[:num_features]]
    top_negative = [{f: v} for f, v in sorted_features[-num_features:]]

    # ---- 5. Format explanation ----
    explanation = {
        # "sample_index": sample_idx,
        "predicted_class": pred_label,   # readable label
        "predicted_probabilities": {class_names.get(i, str(i)): float(p) 
                                    for i, p in enumerate(proba)},
        "top_positive_features": top_positive,
        "top_negative_features": top_negative
    }

    return explanation


# Example usage
explanation_dict = explain_random_sample(model_pruned_c, X_test)
print(explanation_dict)

{'predicted_class': 'early', 'predicted_probabilities': {'early': 0.6621897595326607, 'on_time': 0.10552174563316331, 'delay': 0.23228849483417613}, 'top_positive_features': [{'performance_score_order_full_location': 0.1572278601255895}, {'shipping_mode': 0.07974076432721984}], 'top_negative_features': [{'performance_score_shipping_dayofweek': -0.010248558922217066}, {'order_shipping_time': -0.010759849989984575}]}


In [25]:
"The shipment is likely to be early (66% probability). There is a smaller chance that it will be on time (11%) or delayed (23%). "
"The main reasons are the strong influence of the order’s location and the shipping mode, which increased the chances of early delivery. "
"However, factors like shipping day of the week and shipping time slightly decreased the probability."

'However, factors like shipping day of the week and shipping time slightly decreased the probability.'